## Install and Import

In [1]:
!pip install pandas numpy matplotlib plotly openpyxl -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

## Uploading the Data

## Load and Inspect the RAW Schema

In [7]:
df = pd.read_csv("../data/raw/monthly_ev_sales.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

Shape: (46870, 24)

Columns:
['Area', 'ISO 3 code', 'Area type', 'Date', 'Data frequency', 'Drivetrain', 'Is aggregate drivetrain', 'Registrations', 'Registrations YoY change', 'Registrations YoY change (%)', 'Share of registrations (%)', 'Year-to-date registrations', 'Year-to-date share of registrations (%)', 'Rolling 3-month registrations', 'Rolling 3-month share of registrations (%)', 'Rolling 12-month registrations', 'Rolling 12-month share of registrations (%)', 'Continent', 'Ember region', 'EU member', 'OECD member', 'G20 member', 'G7 member', 'ASEAN member']

First 5 rows:


,Area,ISO 3 code,Area type,Date,Data frequency,Drivetrain,Is aggregate drivetrain,Registrations,Registrations YoY change,Registrations YoY change (%),...,Rolling 3-month share of registrations (%),Rolling 12-month registrations,Rolling 12-month share of registrations (%),Continent,Ember region,EU member,OECD member,G20 member,G7 member,ASEAN member
0,Albania,ALB,Country or economy,2019-01-01,Monthly,Battery electric,False,1.0,NaN,NaN,...,NaN,NaN,NaN,Europe,Europe,False,False,False,False,False
1,Albania,ALB,Country or economy,2019-01-01,Monthly,Plug-in hybrid,False,0.0,NaN,NaN,...,NaN,NaN,NaN,Europe,Europe,False,False,False,False,False
2,Albania,ALB,Country or economy,2019-01-01,Monthly,Hybrid,False,4.0,NaN,NaN,...,NaN,NaN,NaN,Europe,Europe,False,False,False,False,False
3,Albania,ALB,Country or economy,2019-01-01,Monthly,Petrol,False,412.0,NaN,NaN,...,NaN,NaN,NaN,Europe,Europe,False,False,False,False,False
4,Albania,ALB,Country or economy,2019-01-01,Monthly,Diesel,False,2416.0,NaN,NaN,...,NaN,NaN,NaN,Europe,Europe,False,False,False,False,False



Data types:
Area                                               str
ISO 3 code                                         str
Area type                                          str
Date                                               str
Data frequency                                     str
Drivetrain                                         str
Is aggregate drivetrain                           bool
Registrations                                  float64
Registrations YoY change                       float64
Registrations YoY change (%)                   float64
Share of registrations (%)                     float64
Year-to-date registrations                       int64
Year-to-date share of registrations (%)        float64
Rolling 3-month registrations                  float64
Rolling 3-month share of registrations (%)     float64
Rolling 12-month registrations                 float64
Rolling 12-month share of registrations (%)    float64
Continent                                          s

## Check Dimensions and Coverage

In [8]:
# Parse date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Number of unique areas:", df["Area"].nunique())
print("Area types:", df["Area type"].unique())
print("Drivetrains:", df["Drivetrain"].unique())
print("Aggregate flag values:", df["Is aggregate drivetrain"].unique())

# Latest observation per country
latest = (
    df[df["Is aggregate drivetrain"] == False]
    .groupby("Area")["Date"]
    .max()
    .sort_values(ascending=False)
)
print("\nLatest observation date per country (top 10):")
print(latest.head(10))

Date range: 2005-01-01 00:00:00 to 2026-10-01 00:00:00
Number of unique areas: 61
Area types: <StringArray>
['Country or economy', 'Region']
Length: 2, dtype: str
Drivetrains: <StringArray>
[           'Battery electric',              'Plug-in hybrid',
                      'Hybrid',                      'Petrol',
                      'Diesel',                       'Other',
 'Electric vehicle (BEV+PHEV)',             'All drivetrains',
        'Ethanol/petrol blend',    'Combustion (unspecified)']
Length: 10, dtype: str
Aggregate flag values: [False  True]

Latest observation date per country (top 10):
Area
Albania       2026-10-01
India         2026-10-01
Israel        2026-10-01
Italy         2026-10-01
Japan         2026-10-01
Australia     2026-10-01
Malaysia      2026-10-01
Nepal         2026-10-01
Netherlands   2026-10-01
New Zealand   2026-10-01
Name: Date, dtype: datetime64[us]


This tells us which countries have the freshest data and which series end earlier—exactly what Ember warns about

## Separate Aggregates from Individual Drivetrains

In [9]:
# Training table: exclude aggregate drivetrain rows
train_df = df[df["Is aggregate drivetrain"] == False].copy()

# Aggregate table: for share calculations only
agg_df = df[df["Is aggregate drivetrain"] == True].copy()

print("Training rows:", len(train_df))
print("Aggregate rows:", len(agg_df))
print("Drivetrains in training table:", train_df["Drivetrain"].unique())

Training rows: 34092
Aggregate rows: 12778
Drivetrains in training table: <StringArray>
[        'Battery electric',           'Plug-in hybrid',
                   'Hybrid',                   'Petrol',
                   'Diesel',                    'Other',
     'Ethanol/petrol blend', 'Combustion (unspecified)']
Length: 8, dtype: str


## Missingness and Sructutal Zones

In [10]:
# Count zero-registration rows per country–drivetrain series
series_summary = (
    train_df.groupby(["Area", "ISO 3 code", "Drivetrain"])
    .agg(
        n_obs=("Registrations", "size"),
        n_nonzero=("Registrations", lambda x: (x > 0).sum()),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .reset_index()
)

# Series with zero non-zero observations are structurally absent
structurally_absent = series_summary[series_summary["n_nonzero"] == 0]
print("Structurally absent series (all zeros):", len(structurally_absent))

# Series with some zeros but at least one non-zero are reporting zeros
reporting_zeros = series_summary[
    (series_summary["n_nonzero"] > 0) & (series_summary["n_obs"] > series_summary["n_nonzero"])
]
print("Series with reporting zeros:", len(reporting_zeros))

# Series with fewer than 24 observations (too short for lag-12 features reliably)
short_series = series_summary[series_summary["n_obs"] < 24]
print("Series with <24 observations:", len(short_series))

Structurally absent series (all zeros): 12
Series with reporting zeros: 66
Series with <24 observations: 4


In [12]:
series_summary.to_csv("ember_series_quality_summary.csv", index=False)

## Quick Visual Sanity Check

In [ ]:
import plotly.express as px

# Choose one country and one drivetrain to verify the time series looks plausible
sample = train_df[(train_df["Area"] == "Germany") & (train_df["Drivetrain"] == "Battery electric")]

fig = px.line(sample, x="Date", y="Registrations", title="Germany — Battery Electric Registrations")
fig.show()

## Findings

- Row Count: 46870
- Countries: 59
- Date Range: 2005-01-01 00:00:00 to 2026-10-01 00:00:00
- Structural Zero Count: 66
- Rows total: 46870
- Training rows (non-aggregate): 34092
- Aggregate rows: 12778
- Date range: 2005-01-01 00:00:00 to 2026-10-01 00:00:00
- Countries: 59
- Drivetrains: ['Battery electric', 'Combustion (unspecified)', 'Diesel', 'Ethanol/petrol blend', 'Hybrid', 'Other', 'Petrol', 'Plug-in hybrid']

- Oldest latest-observation dates:
Area
* Taiwan (China)   2026-06-01
* South Africa     2026-06-01
* Canada           2026-06-01
* Brazil           2026-07-01
* Türkiye          2026-08-01
Name: Date, dtype: datetime64[us]


In [15]:
# 1. Exact country vs region breakdown
print("Countries:", df[df['Area type']=='Country or economy']['Area'].nunique())
print("Regions:", df[df['Area type']=='Region']['Area'].nunique())
print("Region names:", df[df['Area type']=='Region']['Area'].unique().tolist())
 
# 2. Total number of series (this is missing from your output!)
print("Total country-drivetrain series:", len(series_summary))
 
# 3. Data frequency
print("Data frequency values:", df['Data frequency'].unique())
 
# 4. Membership dtype check
for col in ['EU member','OECD member','G20 member','G7 member','ASEAN member']:
    print(col, "->", df[col].dtype, "| values:", df[col].unique()[:5])
 
# 5. NaN check on target
print("NaN Registrations:", train_df['Registrations'].isna().sum())
 
# 6. Duplicate check on primary key
print("Duplicate (ISO3, Date, Drivetrain) rows:",
      train_df.duplicated(['ISO 3 code','Date','Drivetrain']).sum())
 
# 7. Latest observation distribution
latest_all = train_df.groupby('Area')['Date'].max()
print("\nCountries ending in 2026-10:", (latest_all == '2026-10-01').sum())
print("Countries ending before 2026-10:", (latest_all < '2026-10-01').sum())
print("\nThe early-ending countries:")
print(latest_all[latest_all < '2026-10-01'].sort_values())
 
# 8. Name the 12 structurally absent series
print("\n=== Structurally absent series ===")
print(structurally_absent[['Area','Drivetrain']].to_string())
 
# 9. Name the 4 short series
print("\n=== Series with <24 observations ===")
print(short_series[['Area','Drivetrain','n_obs']].to_string())

Countries: 59
Regions: 2
Region names: ['EU', 'World']
Total country-drivetrain series: 316
Data frequency values: <StringArray>
['Monthly', 'Quarterly']
Length: 2, dtype: str
EU member -> object | values: [False True nan]
OECD member -> object | values: [False True nan]
G20 member -> object | values: [False True nan]
G7 member -> object | values: [False True nan]
ASEAN member -> object | values: [False nan True]
NaN Registrations: 284
Duplicate (ISO3, Date, Drivetrain) rows: 90

Countries ending in 2026-10: 31
Countries ending before 2026-10: 30

The early-ending countries:
Area
Canada                     2026-06-01
Taiwan (China)             2026-06-01
South Africa               2026-06-01
Brazil                     2026-07-01
Türkiye                    2026-08-01
Hong Kong (SAR of China)   2026-08-01
Ukraine                    2026-08-01
Austria                    2026-09-01
Thailand                   2026-09-01
South Korea                2026-09-01
Slovenia                   2026-0